# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hardik144/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I will use Random Forest Regression to predict clicks in the most recent 30-day period using information available from the previous 30-day period and other appropriate content features.

I selected this method because it can model nonlinear relationships and interactions between features without requiring a linear relationship between every input and the target.

I will compare the model against a simple Week-4 baseline that predicts recent clicks using the previous 30-day click count. Both approaches will be evaluated on the same held-out observations using MAE and RMSE.

I will exclude identifiers and any features that contain information from the target period or are derived from the target. The model will be treated as a decision-support experiment, not proof that changing a page will increase clicks.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [7]:

import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("Available columns:")
print(df.columns.tolist())

display(df.head())

Dataset loaded successfully!
Dataset shape: (30000, 44)
Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [8]:

# SECTION 2: SPLIT DESIGN

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit

# Check that the dataset exists
print("Dataset shape:", df.shape)
print("Available columns:")
print(df.columns.tolist())

# Confirm the target and baseline columns
target_col = "clicks_last_30d"
baseline_col = "clicks_prev_30d"

required = [target_col, baseline_col]

missing = [col for col in required if col not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Check the actual dataset column names before proceeding."
    )

# Remove rows without a target
model_df = df.dropna(subset=[target_col]).copy()

# Use a group-based split if client_id is available.
# This keeps the same client out of both train and test sets.
if "client_id" in model_df.columns:
    groups = model_df["client_id"].fillna("UNKNOWN_CLIENT")

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.20,
        random_state=42
    )

    train_idx, test_idx = next(
        splitter.split(model_df, groups=groups)
    )

    train_df = model_df.iloc[train_idx].copy()
    test_df = model_df.iloc[test_idx].copy()

    print("Split design: grouped by client_id")

else:
    train_df, test_df = train_test_split(
        model_df,
        test_size=0.20,
        random_state=42
    )

    print("Split design: random 80/20 holdout")

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Target:", target_col)

# Check for client overlap when grouping was possible
if "client_id" in model_df.columns:
    overlap = set(train_df["client_id"].dropna()) & set(
        test_df["client_id"].dropna()
    )
    print("Client overlap:", len(overlap))

Dataset shape: (30000, 44)
Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Split design: grouped by client_id
Training rows: 23837
Testing rows: 6163
Target: clicks_last_30d
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:

# SECTION 3: TRAIN + COMPARE VS MY BASELINE

import json
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Use only features that exist in the dataset.
# These are available before the target 30-day period.
candidate_features = [
    "clicks_prev_30d",
    "impressions_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "search_volume",
    "competition",
    "cp",
    "content_type",
    "main_intent"
]

features = [
    col for col in candidate_features
    if col in model_df.columns
]

if not features:
    raise ValueError("No eligible features were found.")

print("Features used:", features)

# Separate inputs and target
X_train = train_df[features].copy()
X_test = test_df[features].copy()

y_train = train_df[target_col].astype(float)
y_test = test_df[target_col].astype(float)

# Detect numerical and categorical features
numeric_features = X_train.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_features = [
    col for col in features if col not in numeric_features
]

# Preprocess numeric and categorical data
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Random Forest model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=150,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
model.fit(X_train, y_train)

# Predictions on the held-out test set
model_predictions = np.maximum(model.predict(X_test), 0)

# Week-4 baseline: previous-period clicks
baseline_predictions = np.maximum(
    pd.to_numeric(
        test_df[baseline_col], errors="coerce"
    ).fillna(0).to_numpy(dtype=float),
    0
)

# Calculate metrics on exactly the same test observations
model_mae = mean_absolute_error(y_test, model_predictions)
model_rmse = np.sqrt(mean_squared_error(y_test, model_predictions))

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

comparison = pd.DataFrame([
    {
        "method": "Week-4 baseline",
        "MAE": baseline_mae,
        "RMSE": baseline_rmse
    },
    {
        "method": "Random Forest",
        "MAE": model_mae,
        "RMSE": model_rmse
    }
])

print("MODEL VS BASELINE")
display(comparison)

# Keep predictions for the error analysis
evaluation_df = test_df.copy()
evaluation_df["actual_clicks"] = y_test.to_numpy()
evaluation_df["baseline_prediction"] = baseline_predictions
evaluation_df["model_prediction"] = model_predictions

evaluation_df["baseline_absolute_error"] = np.abs(
    evaluation_df["actual_clicks"]
    - evaluation_df["baseline_prediction"]
)

evaluation_df["model_absolute_error"] = np.abs(
    evaluation_df["actual_clicks"]
    - evaluation_df["model_prediction"]
)

# Save a small metrics receipt, not the underlying data
output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

metrics = {
    "target": target_col,
    "features": features,
    "split": "80/20 holdout; grouped by client_id when available",
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "baseline_mae": float(baseline_mae),
    "baseline_rmse": float(baseline_rmse),
    "model_mae": float(model_mae),
    "model_rmse": float(model_rmse)
}

with open(output_dir / "w05_model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Metrics saved to work/outputs/w05_model_metrics.json")

Features used: ['clicks_prev_30d', 'impressions_prev_30d', 'sessions_prev_30d', 'content_age_days', 'search_volume', 'competition', 'content_type', 'main_intent']
MODEL VS BASELINE


,method,MAE,RMSE
0,Week-4 baseline,1.897939,10.267310
1,Random Forest,1.909690,12.609042


Metrics saved to work/outputs/w05_model_metrics.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The Random Forest achieved a test MAE of [insert model MAE] and a test RMSE of [insert model RMSE]. The Week-4 baseline achieved a test MAE of [insert baseline MAE] and a test RMSE of [insert baseline RMSE].

The comparison shows that [describe which method had lower errors, or state that the results were similar]. The largest errors occurred in observations with [describe the pattern visible in the error table].

The permutation-importance results indicate that [insert the most important observed features]. These results describe predictive associations in this dataset; they do not establish that changing a feature will cause more clicks.

The model has limitations, including the selected features, the holdout design, and the possibility that patterns in the test set differ from future observations. The baseline remains useful because it provides a simple reference for judging whether the added modeling complexity was worthwhile.


In [10]:

# SECTION 4: ERRORS AND INTERPRETATION

# Show the observations with the largest model errors.
# Avoid displaying client identifiers or private information.

error_columns = [
    "actual_clicks",
    "baseline_prediction",
    "model_prediction",
    "baseline_absolute_error",
    "model_absolute_error"
]

print("10 largest model errors:")
display(
    evaluation_df.nlargest(
        10, "model_absolute_error"
    )[error_columns]
)

# Compare average absolute error by actual-click bucket
evaluation_df["actual_click_bucket"] = pd.cut(
    evaluation_df["actual_clicks"],
    bins=[-np.inf, 0, 5, 20, 100, np.inf],
    labels=["0", "1-5", "6-20", "21-100", "100+"]
)

error_summary = (
    evaluation_df.groupby(
        "actual_click_bucket",
        observed=False
    )
    .agg(
        n=("actual_clicks", "size"),
        model_MAE=("model_absolute_error", "mean"),
        baseline_MAE=("baseline_absolute_error", "mean")
    )
    .reset_index()
)

print("ERRORS BY ACTUAL-CLICK BUCKET")
display(error_summary)

# Permutation importance on the held-out test set
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": features,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std
}).sort_values("importance_mean", ascending=False)

print("PERMUTATION IMPORTANCE")
display(importance_df)

# A positive value means shuffling the feature increased error
# on average. Importance is predictive, not causal.

10 largest model errors:


,actual_clicks,baseline_prediction,model_prediction,baseline_absolute_error,model_absolute_error
21565,1176.0,1440.0,482.022804,264.0,693.977196
18870,501.0,53.0,47.792006,448.0,453.207994
2365,214.0,441.0,383.580053,227.0,169.580053
21819,548.0,662.0,393.995530,114.0,154.004470
28335,13.0,169.0,148.697008,156.0,135.697008
26413,74.0,283.0,192.772791,209.0,118.772791
23446,162.0,76.0,50.649971,86.0,111.350029
26946,179.0,91.0,78.455721,88.0,100.544279
23460,181.0,114.0,87.159187,67.0,93.840813
15353,151.0,65.0,60.514249,86.0,90.485751


ERRORS BY ACTUAL-CLICK BUCKET


,actual_click_bucket,n,model_MAE,baseline_MAE
0,0,4070,0.257147,0.196314
1,1-5,1366,1.400724,1.666911
2,6-20,488,5.252508,5.762295
3,21-100,211,16.838743,16.312796
4,100+,28,96.187183,84.535714


PERMUTATION IMPORTANCE


,feature,importance_mean,importance_std
0,clicks_prev_30d,3.510454,0.046071
1,impressions_prev_30d,0.242242,0.012769
2,sessions_prev_30d,0.049445,0.004773
3,content_age_days,0.005195,0.005723
5,competition,0.004483,0.001975
7,main_intent,0.003489,0.001191
4,search_volume,0.000050,0.001768
6,content_type,0.000000,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.